In [1]:
import pickle
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer


def load_pickle_to_dataframe(file_path: str) -> pd.DataFrame:
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    if isinstance(data, pd.DataFrame):
        return data
    elif isinstance(data, dict):
        return pd.DataFrame.from_dict(data)

    elif isinstance(data, list) and all(isinstance(item, dict) for item in data):
        return pd.DataFrame(data)

    elif isinstance(data, np.ndarray):
        return pd.DataFrame(data)
    else:
        raise ValueError("Unsupported data type in pickle file")

/Users/mathisbenyahia/llm-for-financial-service/.venv/lib/python3.13/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [ ]:

ent = load_pickle_to_dataframe("stocks_yahoo.pkl")

In [ ]:
ent

In [ ]:
ent[ent["longName"].str.contains("LVMH",na=False)]

In [ ]:
#étape 1 : pour chaque longName, on garde la avgvolyme3m max 
ent = ent.sort_values(by=["longName","avgvolume3m"],ascending=[True,False])
# étape 2 : on supprime toutes les lignes non traités dans les 3 mois et sans market_cap
ent = ent.dropna(subset=["avgvolume3m","market_cap"])
# étape 3: on supprime tous les doublons de longName (on garde le premier)
ent = ent.sort_values(by=["longName","avgvolume3m"],ascending=[True,False])
ent = ent.drop_duplicates(subset=["longName"],keep="first")
# étape 4 : on construit la colonne shortTicker
ent["shortTicker"] = ent["ticker"].str.split(".").str[0]
# étape 5 : on refiltre pour ne garder qu'une instance
ent = ent.sort_values(by=["shortTicker","avgvolume3m"],ascending=[True,False])
#ent = ent.drop_duplicates(subset=["shortTicker"],keep="first")
#étape 6 = p, refiltre pour ne garde qu'une instance de shortName
ent = ent.sort_values(by=["shortName","avgvolume3m"],ascending=[True,False])
ent = ent.drop_duplicates(subset=["shortName"],keep="first")

ent

In [ ]:
ent[ent["longName"].str.contains("LVMH",na=False)]

In [ ]:
from yfinance import Ticker
t= Ticker("MC.PA").info
print(t["website"])

In [ ]:

#on ajout des infos au dataframe : industry,longBusinessSummary
def fetch_infos(row):
    t = Ticker(row["ticker"]).info
    return pd.Series([t.get("industry",""),t.get("industry","")])

ent[["industry","longBusinessSummary"]] = ent.apply(fetch_infos,axis=1)

In [ ]:
ent.to_pickle("entreprises_cotees_enrichies.pickle")

# PARTIE ENTRAINEMENT DE L ANALYSEUR SEMANTIQUE

In [2]:
ent = load_pickle_to_dataframe("stocks_yahoo_enriched.pickle")

In [10]:
# étape 3 : on supprime tous les doublons de longName (on garde le premier)
ent = ent.sort_values(by=["website","avgvolume3m"],ascending=[True,False])
ent = ent.drop_duplicates(subset=["website"],keep="first")


In [11]:
ent[ent["longName"].str.contains("LVMH",na=False)]

,ticker,longName,isin,shortName,market_cap,region,quoteType,exchange,exchangeFullName,avgvolume3m,shortTicker,industry,summary,website
4885,MC.PA,"LVMH Moët Hennessy - Louis Vuitton, Société Eu...",NaN,LVMH,$227.43B,US,EQUITY,PAR,Paris,468324.0,MC,Luxury Goods,"LVMH Moët Hennessy - Louis Vuitton, Société Eu...",https://www.lvmh.com


In [13]:
ent["richString"] = "Ticker : " + ent["ticker"] + " | LongName : " + ent["longName"] + " | ShortName : " + ent["shortName"] + " | Industry : " + ent["industry"] + " | Summary : " + ent["summary"].str[:500]

In [ ]:
ent

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
richString = ent["richString"].to_list()
print(richString)

In [ ]:
embeddings = model.encode(richString,convert_to_tensor=True,show_progress_bar=True)

In [ ]:
import torch
torch.save(embeddings,"company_embeddings.pt")